# UHM Coefficient Distribution Analysis

Projects all samples in `/data/datasets/UHM_generated_data/` back onto the UHM PCA model
and measures the coefficient distribution for the first 50 eigenmodes.

Each normalised coefficient `c_i = (U_i^T (x - μ)) / sqrt(λ_i)` should be N(0,1) if the data was generated from the model.

In [ ]:
import pickle
import numpy as np
import open3d as o3d
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Load model

In [ ]:
MODEL_PATH = '/data/UHM_models/head_model_global_align_no_mouth_and_eyes.pkl'
DATA_DIR   = Path('/data/datasets/UHM_generated_data/')
N_MODES    = 50

with open(MODEL_PATH, 'rb') as f:
    uhm = pickle.load(f)

mu  = uhm['Mean'].squeeze()          # (3V,)
U   = uhm['Eigenvectors'][:, :N_MODES] # (3V, 50)
lam = uhm['EigenValues'][:N_MODES]    # (50,)  -- variance per mode
sig = np.sqrt(lam)                     # std per mode

n_verts = mu.shape[0] // 3
print(f'Model vertices : {n_verts:,}')
print(f'Modes kept     : {N_MODES}')
print(f'Eigenvalue range: {lam[0]:.4f} … {lam[-1]:.6f}')

In [ ]:
U.dtype

## Project all samples

In [ ]:
pts.dtype

In [ ]:
ply_files = sorted(p for p in DATA_DIR.glob('*.ply') if not p.name.startswith('.'))
print(f'Found {len(ply_files):,} PLY files')

coeffs = np.empty((len(ply_files), N_MODES), dtype=np.float64)

for i, path in enumerate(tqdm(ply_files, desc='Projecting')):
    filname = path.name
    if filname.startswith('._'):
        print(f'Skipping {filname} (_)')
        continue
    pcd = o3d.io.read_point_cloud(str(path))
    pts = np.asarray(pcd.points)  # (V, 3)
    x   = pts.ravel()             # (3V,)
    diff = x - mu                 # (3V,)
    # raw PCA scores
    raw = U.T @ diff              # (50,)
    # normalise by sqrt(eigenvalue) -> should be N(0,1)
    coeffs[i] = raw / sig

print(f'Coefficient matrix: {coeffs.shape}')

## Per-mode statistics

In [ ]:
means  = coeffs.mean(axis=0)
stds   = coeffs.std(axis=0)
# Shapiro-Wilk on a random subsample (max 5 k per mode)
rng    = np.random.default_rng(42)
n_sw   = min(5000, len(coeffs))
idx    = rng.choice(len(coeffs), n_sw, replace=False)
sw_p   = np.array([stats.shapiro(coeffs[idx, k])[1] for k in range(N_MODES)])

print(f"{'Mode':>6}  {'mean':>8}  {'std':>8}  {'SW p-val':>10}")
print('-' * 40)
for k in range(N_MODES):
    flag = '' if sw_p[k] > 0.05 else '  <- non-normal'
    print(f'{k+1:>6}  {means[k]:>8.4f}  {stds[k]:>8.4f}  {sw_p[k]:>10.4f}{flag}')

## Summary: mean and std across modes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
modes = np.arange(1, N_MODES + 1)

ax = axes[0]
ax.bar(modes, means, color='steelblue', alpha=0.8)
ax.axhline(0, color='k', linewidth=1)
ax.set_xlabel('Mode')
ax.set_ylabel('Mean coefficient')
ax.set_title('Per-mode mean  (target: 0)')

ax = axes[1]
ax.bar(modes, stds, color='coral', alpha=0.8)
ax.axhline(1, color='k', linewidth=1, linestyle='--', label='target σ=1')
ax.set_xlabel('Mode')
ax.set_ylabel('Std coefficient')
ax.set_title('Per-mode std  (target: 1)')
ax.legend()

plt.tight_layout()
plt.show()

## Histogram grid — first 50 modes

In [ ]:
ncols = 10
nrows = N_MODES // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.5, nrows * 2.2))
axes = axes.ravel()

x_grid = np.linspace(-4, 4, 200)
gauss  = stats.norm.pdf(x_grid)

for k in range(N_MODES):
    ax = axes[k]
    ax.hist(coeffs[:, k], bins=60, density=True, alpha=0.6,
            color='steelblue', label='data')
    ax.plot(x_grid, gauss, 'r-', linewidth=1.2, label='N(0,1)')
    ax.set_title(f'Mode {k+1}\nμ={means[k]:.2f} σ={stds[k]:.2f}', fontsize=8)
    ax.set_xlim(-5, 5)
    ax.tick_params(labelsize=6)
    ax.set_yticks([])

# hide any unused panels
for k in range(N_MODES, len(axes)):
    axes[k].set_visible(False)

handles = [plt.Rectangle((0,0),1,1, fc='steelblue', alpha=0.6), plt.Line2D([0],[0], color='r')]
fig.legend(handles, ['data', 'N(0,1)'], loc='lower right', fontsize=9)
fig.suptitle('Normalised PCA coefficients — first 50 modes', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## QQ-plot grid

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.2, nrows * 2.2))
axes = axes.ravel()

for k in range(N_MODES):
    ax = axes[k]
    (osm, osr), (slope, intercept, _) = stats.probplot(coeffs[:, k], dist='norm')
    ax.scatter(osm, osr, s=1, alpha=0.3, color='steelblue')
    ax.plot(osm, slope * np.array(osm) + intercept, 'r-', linewidth=1)
    ax.set_title(f'Mode {k+1}', fontsize=8)
    ax.tick_params(labelsize=6)

for k in range(N_MODES, len(axes)):
    axes[k].set_visible(False)

fig.suptitle('QQ-plots vs N(0,1) — first 50 modes', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Heatmap: coefficient correlation across modes

In [ ]:
corr = np.corrcoef(coeffs.T)  # (50, 50)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
plt.colorbar(im, ax=ax)
ax.set_title('Cross-mode correlation (should be ~diagonal)')
ax.set_xlabel('Mode')
ax.set_ylabel('Mode')
ticks = np.arange(0, N_MODES, 5)
ax.set_xticks(ticks)
ax.set_xticklabels(ticks + 1)
ax.set_yticks(ticks)
ax.set_yticklabels(ticks + 1)
plt.tight_layout()
plt.show()

off_diag = corr[np.triu_indices(N_MODES, k=1)]
print(f'Off-diagonal correlation — mean: {off_diag.mean():.4f}, max abs: {np.abs(off_diag).max():.4f}')

## Aggregate normality summary

In [ ]:
n_normal = (sw_p > 0.05).sum()
print(f'Modes passing Shapiro-Wilk (p>0.05): {n_normal}/{N_MODES}')
print(f'Global mean of all coefficients : {coeffs.mean():.4f}  (target: 0)')
print(f'Global std  of all coefficients : {coeffs.std():.4f}  (target: 1)')

fig, ax = plt.subplots(figsize=(9, 3))
ax.scatter(range(1, N_MODES+1), sw_p, c=(sw_p > 0.05), cmap='RdYlGn', vmin=0, vmax=1, s=40, zorder=3)
ax.axhline(0.05, color='grey', linestyle='--', linewidth=1, label='p=0.05')
ax.set_xlabel('Mode')
ax.set_ylabel('Shapiro-Wilk p-value')
ax.set_title('Normality test per mode (green = normal)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# compute width/height/depth from existing per_file_stats and plot distributions
# per_file_stats entries: (name, n, minx,miny,minz, maxx,maxy,maxz, meanx,meany,meanz, stdx,stdy,stdz, medianx,mediany,medianz)

if not per_file_stats:
    raise RuntimeError("per_file_stats is empty - run the stats collection cell first")

dims = np.array([[row[5] - row[2],  # width = maxx - minx
                  row[6] - row[3],  # height = maxy - miny
                  row[7] - row[4]]  # depth = maxz - minz
                 for row in per_file_stats], dtype=np.float64)

names = np.array([row[0] for row in per_file_stats])
n_files = dims.shape[0]

# summary statistics
def summarize(a):
    return {
        'count': int(np.sum(~np.isnan(a))),
        'mean': float(np.nanmean(a)),
        'median': float(np.nanmedian(a)),
        'std': float(np.nanstd(a)),
        'min': float(np.nanmin(a)),
        'max': float(np.nanmax(a)),
        'p10': float(np.nanpercentile(a, 10)),
        'p90': float(np.nanpercentile(a, 90))
    }

width, height, depth = dims[:,0], dims[:,1], dims[:,2]
summ = {k: summarize(v) for k, v in zip(('width','height','depth'), (width, height, depth))}

print(f"Files analysed: {n_files}\n")
for k in ('width','height','depth'):
    s = summ[k]
    print(f"{k:6} | mean={s['mean']:.6f}, median={s['median']:.6f}, std={s['std']:.6f}, min={s['min']:.6f}, max={s['max']:.6f}")

# plots: histogram + KDE for each axis
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
labels = ['Width (x)', 'Height (y)', 'Depth (z)']
colors = ['steelblue', 'coral', 'seagreen']

for i, ax in enumerate(axes):
    data = dims[:, i]
    data = data[~np.isnan(data)]
    ax.hist(data, bins=80, density=True, color=colors[i], alpha=0.6)
    try:
        kde = stats.gaussian_kde(data)
        xs = np.linspace(np.min(data), np.max(data), 300)
        ax.plot(xs, kde(xs), 'r-', linewidth=1)
    except Exception:
        pass
    ax.axvline(np.mean(data), color='k', linestyle='--', linewidth=1)
    ax.axvline(np.median(data), color='k', linestyle=':', linewidth=1)
    ax.set_title(f"{labels[i]}  (mean={np.mean(data):.3f}, med={np.median(data):.3f})")
    ax.set_xlabel('meters')
    ax.set_yticks([])

plt.tight_layout()
plt.show()

---
## Face-region spatial statistics
Uses `light_model_face_mask.pkl` to restrict each scan to face vertices only, then reports bounding-box extents (width / height / depth) and point-cloud density across all samples.

In [ ]:
MASK_PATH = '/data/UHM_models/Landmarks and masks/light_model_face_mask.pkl'

with open(MASK_PATH, 'rb') as f:
    face_mask = pickle.load(f)  # (V,) bool

print(f'Total head vertices : {len(face_mask):,}')
print(f'Face-region vertices: {face_mask.sum():,}  ({face_mask.mean()*100:.1f}%)')

# collect spatial stats per scan
records = []
for path in tqdm(ply_files, desc='Face stats'):
    pcd  = o3d.io.read_point_cloud(str(path))
    pts  = np.asarray(pcd.points)       # (V, 3)
    face = pts[face_mask]               # (n_face, 3)
    mn, mx = face.min(axis=0), face.max(axis=0)
    records.append({
        'width':  float(mx[0] - mn[0]),
        'height': float(mx[1] - mn[1]),
        'depth':  float(mx[2] - mn[2]),
        'n_verts': int(face_mask.sum()),
    })

import pandas as pd
df_face = pd.DataFrame(records)
print('\nFace-region summary (metres):')
display(df_face[['width', 'height', 'depth']].describe().round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Face-region bounding-box extents (metres)', fontsize=13, fontweight='bold')

dim_labels = {'width': 'X extent', 'height': 'Y extent', 'depth': 'Z extent'}
palette = ['#4C72B0', '#DD8452', '#55A868']

for ax, (dim, label), color in zip(axes, dim_labels.items(), palette):
    data = df_face[dim]
    ax.hist(data, bins=40, color=color, edgecolor='white', linewidth=0.5, density=True, alpha=0.8)
    try:
        kde = stats.gaussian_kde(data)
        xs = np.linspace(data.min(), data.max(), 300)
        ax.plot(xs, kde(xs), 'k-', linewidth=1.4)
    except Exception:
        pass
    ax.axvline(data.mean(),   color='black', linewidth=1.2, linestyle='--', label=f'mean={data.mean():.4f}')
    ax.axvline(data.median(), color='grey',  linewidth=1.0, linestyle=':',  label=f'med={data.median():.4f}')
    ax.set_title(f'{dim.capitalize()}  ({label})')
    ax.set_xlabel('metres')
    ax.set_yticks([])
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# scatter: width vs height, width vs depth
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Face-region extent correlations', fontsize=12)

for ax, (xa, ya) in zip(axes, [('width', 'height'), ('width', 'depth')]):
    ax.scatter(df_face[xa], df_face[ya], s=4, alpha=0.4, color='#4C72B0')
    ax.set_xlabel(xa); ax.set_ylabel(ya)
    ax.grid(linestyle='--', alpha=0.4)
    r = np.corrcoef(df_face[xa], df_face[ya])[0, 1]
    ax.set_title(f'r = {r:.3f}')

plt.tight_layout()
plt.show()

### Example face scans (masked point clouds)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

N_EXAMPLES = 6  # number of scans to show
sample_files = ply_files[:N_EXAMPLES]

fig = plt.figure(figsize=(N_EXAMPLES * 3.5, 10))
fig.suptitle('Example face scans (masked)', fontsize=13, fontweight='bold')

for col, path in enumerate(sample_files):
    pcd  = o3d.io.read_point_cloud(str(path))
    pts  = np.asarray(pcd.points)
    face = pts[face_mask]           # (n_face, 3)
    x, y, z = face[:, 0], face[:, 1], face[:, 2]

    # 3-D view
    ax3d = fig.add_subplot(3, N_EXAMPLES, col + 1, projection='3d')
    ax3d.scatter(x, y, z, s=0.5, c=z, cmap='viridis', alpha=0.6)
    ax3d.set_title(path.stem[:14], fontsize=7)
    ax3d.set_xlabel('X', fontsize=6); ax3d.set_ylabel('Y', fontsize=6); ax3d.set_zlabel('Z', fontsize=6)
    ax3d.tick_params(labelsize=5)

    # front XY
    ax_xy = fig.add_subplot(3, N_EXAMPLES, N_EXAMPLES + col + 1)
    ax_xy.scatter(x, y, s=0.3, c=z, cmap='viridis', alpha=0.5)
    ax_xy.set_aspect('equal'); ax_xy.set_xlabel('X', fontsize=6); ax_xy.set_ylabel('Y', fontsize=6)
    ax_xy.set_title('front XY', fontsize=7); ax_xy.tick_params(labelsize=5)

    # side ZY
    ax_zy = fig.add_subplot(3, N_EXAMPLES, 2 * N_EXAMPLES + col + 1)
    ax_zy.scatter(z, y, s=0.3, c=x, cmap='plasma', alpha=0.5)
    ax_zy.set_aspect('equal'); ax_zy.set_xlabel('Z', fontsize=6); ax_zy.set_ylabel('Y', fontsize=6)
    ax_zy.set_title('side ZY', fontsize=7); ax_zy.tick_params(labelsize=5)

plt.tight_layout()
plt.show()